In [ ]:
import numpy as np # linear algebra
import pandas as pd
import os
import itertools
from keras.layers import Attention
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D, BatchNormalization, Bidirectional, GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import psutil
import os
#from google.colab import drive
#drive.mount('/content/drive')
from imblearn.over_sampling import BorderlineSMOTE,SMOTE
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

import time
from sklearn.metrics import classification_report, confusion_matrix,precision_recall_fscore_support, roc_auc_score
from imblearn.metrics import geometric_mean_score
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')



In [ ]:

# Function to report memory (MB)
def memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

# Initial memory
mem_start = memory_usage_mb()
print(f"Initial memory usage: {mem_start:.2f} MB")
# =====================================================
# 1. Load and combine CSV files
# =====================================================
# Load and Preprocess Data
# ----------------------
csv_dir = "/content/drive/MyDrive/IoT-23"
dfs = []



filename = f"{csv_dir}/iot23_combined_new.csv"

try:
    print(f"Loading: {filename}")
    df_combined = df_combined = pd.read_csv(filename)
    print("DataFrame shape:", df_combined.shape)
except FileNotFoundError:
    raise FileNotFoundError(f"File not found: {filename}")
mem_after_data = memory_usage_mb()
print(f"Memory after data load: {mem_after_data:.2f} MB")


In [ ]:
dataset_name='IoT23'

In [ ]:
label = 'label'

In [ ]:
print("Original Classes")
class_counts = df_combined[label].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']

display(class_counts)

In [ ]:
print("No of Samples before:")
print(df_combined.shape[0])

In [ ]:

print("ORIGINAL FEATURE INFO")

feature_info = pd.DataFrame({
    'Feature': df_combined.columns,
    'Data Type': df_combined.dtypes.astype(str),
    'Missing': df_combined.isna().sum(),
    'Missing %': (df_combined.isna().mean() * 100).round(2),
    'Unique': df_combined.nunique(),
    'Mean': df_combined.mean(numeric_only=True),
    'Std': df_combined.std(numeric_only=True),
    'Min': df_combined.min(numeric_only=True),
    'Max': df_combined.max(numeric_only=True)
})

display(feature_info.round(4))

In [ ]:
print("Total NaNs in dataset: ", df_combined.isna().sum().sum())

In [ ]:
df_combined.head()

In [ ]:

# =====================================================
# 2. Drop irrelevant / identifier columns
# =====================================================
drop_cols = [
    "ts", "uid" ,"Unnamed: 0"       # hardware identifiers
]
df_combined.drop(columns=drop_cols, inplace=True, errors="ignore")
df_combined[label] = df_combined[label].str.replace("PartOfA", "", regex=False)


In [ ]:
# Count occurrences of each label
label_counts = df_combined[label].value_counts()

# Keep only labels with at least 50 samples
valid_labels = label_counts[label_counts >= 50].index

# Filter the DataFrame
df_combined = df_combined[df_combined[label].isin(valid_labels)]

# Check the result
print(df_combined["label"].value_counts())

In [ ]:
df_combined[label] = df_combined[label].str.strip()

In [ ]:
df_combined = df_combined.rename(columns={ label: 'label'})


In [ ]:
print("Retained Classes")
class_counts = df_combined['label'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']

display(class_counts)

In [ ]:
retained_features = pd.DataFrame({
    "Feature": df_combined.columns
})

print("Retained Features")
display(retained_features)

In [ ]:
print("No of Samples After:")
print(df_combined.shape[0])

In [ ]:

# Encode labels
print("Encoding labels...")
label_encoder = LabelEncoder()
df_combined['label'] = label_encoder.fit_transform(df_combined['label'])

class_mapping = {
    cls: int(i)
    for i, cls in enumerate(label_encoder.classes_)
}

mapping_df = pd.DataFrame(
    list(class_mapping.items()),
    columns=['Original Label', 'Encoded Label']
)

print("Label Mapping")
display(mapping_df)

In [ ]:
numeric_columns = df_combined.select_dtypes(include=["number"]).columns
df_combined[numeric_columns] = df_combined[numeric_columns].fillna(df_combined[numeric_columns].median())

# Encode categorical variables
non_numeric_columns = df_combined.select_dtypes(exclude=["number"]).columns
for col in non_numeric_columns:
    df_combined[col] = df_combined[col].astype(str)
    le = LabelEncoder()
    df_combined[col] = le.fit_transform(df_combined[col])

In [ ]:
def scale_dataset(X_train,X_val,X_test,y_train,y_val,y_test):
    # =====================================================
    # 7. Feature scaling
    # =====================================================
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)



    y_train_labels = y_train
    orig_counts = Counter(y_train_labels)

    target_count = 600000
    sampling_strategy = {
        cls: target_count
        for cls, cnt in orig_counts.items()
        if cnt < target_count
    }
    print(sampling_strategy)

    # Apply SMOTE to the preprocessed data (X_train_processed)
    sm = SMOTE(random_state=16, sampling_strategy=sampling_strategy)
    X_train_scaled, y_train = sm.fit_resample(X_train_scaled, y_train_labels)


    plt.figure(figsize=(6, 4))
    sns.countplot(x=y, palette="coolwarm")
    plt.title("Class Distribution ")
    plt.xlabel("Attack Label")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig("multi-IoT23-distribution.png", dpi=700)
    plt.close()

    # =====================================================
    # 9. Split consistency check
    # =====================================================
    split_labels = (
        ["Train"] * len(y_train) +
        ["Validation"] * len(y_val) +
        ["Test"] * len(y_test)
    )
    y_combined = pd.Series(list(y_train) + list(y_val) + list(y_test), name="Class")
    split_df = pd.DataFrame({"Split": split_labels, "Class": y_combined})

    plt.figure(figsize=(8, 5))
    sns.countplot(data=split_df, x="Class", hue="Split")
    plt.title("Split Consistency: Class Distribution Across Splits")
    plt.xlabel("Attack Label")
    plt.ylabel("Count")
    plt.legend(title="Data Split")
    plt.tight_layout()
    plt.savefig("multi-IoT23-split.png", dpi=700)
    plt.close()
    return X_train_scaled,X_val_scaled,X_test_scaled,y_train



In [ ]:
from tensorflow.keras.layers import Input, Bidirectional, GRU, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model

from tensorflow.keras.layers import Input, LSTM, Bidirectional, Attention
from tensorflow.keras.layers import Dense, BatchNormalization, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

def create_model(input_shape, lstm_units, l2_reg,
                         dense1, dense2, n_classes):

    inputs = Input(shape=input_shape)

    x = Bidirectional(
        LSTM(
            lstm_units,
            return_sequences=True,
            kernel_regularizer=l2(l2_reg)
        )
    )(inputs)

    x = Attention()([x, x])

    x = Flatten()(x)

    x = Dense(dense1, activation="relu")(x)
    x = BatchNormalization()(x)

    x = Dense(dense2, activation="relu")(x)

    outputs = Dense(n_classes, activation="softmax")(x)

    model = Model(inputs, outputs)

    return model




def model_complexity(model, input_shape):
    """
    Computes model complexity metrics.

    Parameters
    ----------
    model : tf.keras.Model
        Trained or untrained Keras model.
    input_shape : tuple
        Shape of one input sample (excluding batch size).
        Example: (122,), (32, 32, 3)

    Returns
    -------
    dict
        Total Parameters
        Trainable Parameters
        Non-trainable Parameters
        Model Size (MB)
        FLOPs (GFLOPs)
    """

    # Build model if not already built
    model.build((None, *input_shape))

    # -----------------------------
    # Parameters
    # -----------------------------
    total_params = model.count_params()

    trainable_params = np.sum(
        [tf.keras.backend.count_params(w) for w in model.trainable_weights]
    )

    non_trainable_params = np.sum(
        [tf.keras.backend.count_params(w) for w in model.non_trainable_weights]
    )

    # -----------------------------
    # Model Size
    # -----------------------------
    model_size_bytes = total_params * 4       # float32 = 4 bytes
    model_size_mb = model_size_bytes / (1024 ** 2)

    # -----------------------------
    # FLOPs
    # -----------------------------
    @tf.function
    def forward_pass(x):
        return model(x)

    concrete_func = forward_pass.get_concrete_function(
        tf.TensorSpec([1, *input_shape], tf.float32)
    )

    from tensorflow.python.framework.convert_to_constants import (
        convert_variables_to_constants_v2,
    )

    frozen_func = convert_variables_to_constants_v2(concrete_func)

    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")

        run_meta = tf.compat.v1.RunMetadata()

        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd='op',
            options=opts
        )

    total_flops = flops.total_float_ops
    gflops = total_flops / 1e9

    return {
        "Total Parameters": int(total_params),
        "Trainable Parameters": int(trainable_params),
        "Non-trainable Parameters": int(non_trainable_params),
        "Model Size (MB)": round(model_size_mb, 3),
        "FLOPs": int(total_flops),
        "GFLOPs": round(gflops, 4)
    }

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(history,model_name):
    """
    Plot Training vs Validation Accuracy.

    Parameters
    ----------
    history : keras.callbacks.History
        History object returned by model.fit().

    """

    plt.figure(figsize=(7, 5))

    plt.plot(
        history.history['accuracy'],
        label='Training Accuracy',
        linewidth=2,
        marker='o',
        markersize=4
    )

    plt.plot(
        history.history['val_accuracy'],
        label='Validation Accuracy',
        linewidth=2,
        marker='s',
        markersize=4
    )

    plt.title('Training vs Validation Accuracy '+model_name, fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='lower right')

    plt.tight_layout()



    plt.show()


import matplotlib.pyplot as plt

def plot_loss(history, model_name):
    """
    Plot Training vs Validation Loss.

    Parameters
    ----------
    history : keras.callbacks.History
        History object returned by model.fit().

    """

    plt.figure(figsize=(7, 5))

    plt.plot(
        history.history['loss'],
        label='Training Loss',
        linewidth=2,
        marker='o',
        markersize=4
    )

    plt.plot(
        history.history['val_loss'],
        label='Validation Loss',
        linewidth=2,
        marker='s',
        markersize=4
    )

    plt.title('Training vs Validation Loss '+ model_name, fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')

    plt.tight_layout()



    plt.show()



In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def plot_confusion_matrix(
    y_true,
    y_pred,
    class_names,
    title,
    normalize=False
):

    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype(float) / (
            cm.sum(axis=1)[:, np.newaxis] + 1e-10
        )

    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f" if normalize else "d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()


    plt.show()
    plt.close()


def plot_multiclass_roc(
    y_true,
    y_prob,
    class_names,
    title
):

    y_true_bin = label_binarize(
        y_true,
        classes=np.arange(len(class_names))
    )

    auc_score = roc_auc_score(
        y_true_bin,
        y_prob,
        multi_class="ovr",
        average="macro"
    )

    plt.figure(figsize=(7,6))

    for i, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(
            y_true_bin[:, i],
            y_prob[:, i]
        )

        plt.plot(
            fpr,
            tpr,
            label=f"{class_name} (AUC={roc_auc_score(y_true_bin[:, i], y_prob[:, i]):.4f})"
        )

    plt.plot([0,1],[0,1],'k--')

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title}\nMacro AUC = {auc_score:.4f}")
    plt.legend()
    plt.show()

    return auc_score

In [ ]:
def classwise_metrics(y_true, y_pred, y_prob, class_names):
    """
    Parameters
    ----------
    y_true : array-like
        True labels.
    y_pred : array-like
        Predicted labels.
    y_prob : array-like of shape (n_samples, n_classes)
        Predicted probabilities for each class.
    class_names : list
        Class names.
    """

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    df = pd.DataFrame(report).transpose()

    # Overall accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # One-vs-Rest ROC-AUC
    classes = np.unique(y_true)
    y_true_bin = label_binarize(y_true, classes=classes)

    roc_auc = roc_auc_score(
        y_true_bin,
        y_prob,
        average=None,
        multi_class="ovr"
    )

    # Compute G-Mean, MCC, and FPR for each class
    gmeans = []
    mccs = []
    fprs = []

    for i, cls in enumerate(classes):

        # One-vs-Rest
        yt = (np.array(y_true) == cls).astype(int)
        yp = (np.array(y_pred) == cls).astype(int)

        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()

        # Sensitivity / Recall
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

        # Specificity
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        # False Positive Rate
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        # G-Mean
        gmean = np.sqrt(sensitivity * specificity)

        # MCC
        mcc = matthews_corrcoef(yt, yp)

        gmeans.append(gmean)
        mccs.append(mcc)
        fprs.append(fpr)

    # Keep only class rows
    df = df.loc[
        class_names,
        ['precision', 'recall', 'f1-score']
    ]

    # Rename columns
    df.columns = ['Precision', 'Recall', 'F1']

    # Add metrics
    df['Accuracy'] = accuracy
    df['FPR'] = fprs
    df['ROC_AUC'] = roc_auc
    df['GMean'] = gmeans
    df['MCC'] = mccs

    # Reorder columns
    df = df[
        [
            'Accuracy',
            'Precision',
            'Recall',
            'F1',
            'FPR',
            'ROC_AUC',
            'GMean',
            'MCC'
        ]
    ]

    return df.round(4)
def calculate_metrics(
    y_true,
    y_pred,
    y_prob
):

    # Calculate multiclass FPR using One-vs-Rest
    classes = np.unique(y_true)

    fprs = []
    supports = []

    for cls in classes:

        # One-vs-Rest transformation
        yt = (np.array(y_true) == cls).astype(int)
        yp = (np.array(y_pred) == cls).astype(int)

        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()

        # False Positive Rate
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        fprs.append(fpr)
        supports.append(np.sum(yt))

    # Weighted FPR
    weighted_fpr = np.average(
        fprs,
        weights=supports
    )

    metrics = {

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Precision":
            precision_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "FPR":
            weighted_fpr,

        "ROC_AUC":
            roc_auc_score(
                y_true,
                y_prob,
                average="weighted",
                multi_class="ovr"
            ),

        "GMean":
            geometric_mean_score(
                y_true,
                y_pred
            ),

        "MCC":
            matthews_corrcoef(
                y_true,
                y_pred
            )
    }

    return pd.DataFrame([metrics])
def timing_report(
    train_time,
    test_time,
    predict_time
):

    return pd.DataFrame({

        "Training Time (s)":
            [train_time],

        "Testing Time (s)":
            [test_time],

        "Prediction Time (s)":
            [predict_time],

        "Total Time (s)":
            [
                train_time +
                test_time +
                predict_time
            ]

    })
import tensorflow as tf
import numpy as np


In [ ]:
def evaluate_dataset(
    model,
    X,
    y,
    dataset_name,
    class_names
):

    print(f"\n{'='*60}")
    print(dataset_name.upper())
    print(f"{'='*60}")

    start = time.time()

    y_prob = model.predict(X)

    y_pred = np.argmax(y_prob, axis=1)

    pred_time = time.time() - start

    metrics_df = calculate_metrics(
        y,
        y_pred,
        y_prob
    )

    display(metrics_df.round(4))

    class_df = classwise_metrics(
        y,
        y_pred,
        y_prob,
        class_names
    )

    print("\nClass-wise Performance")
    display(class_df)

    plot_confusion_matrix(
        y,
        y_pred,
        class_names,
        f"{dataset_name} Confusion Matrix"
    )

    plot_confusion_matrix(
        y,
        y_pred,
        class_names,
        f"{dataset_name} Normalized CM",
        normalize=True
    )

    plot_multiclass_roc(
        y,
        y_prob,
        class_names,
        f"{dataset_name} ROC Curve"
    )

    return metrics_df, class_df, pred_time

In [ ]:
import subprocess

def get_gpu_usage():
    result = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=utilization.gpu",
            "--format=csv,noheader,nounits"
        ]
    ).decode("utf-8").strip()

    return float(result)

In [ ]:
def aggregate_fold_metrics(fold_results):

    rows = []

    for fold_idx, result in enumerate(fold_results, start=1):

        for dataset_name, key in [
            ("Train", "train_metrics"),
            ("Validation", "val_metrics"),
            ("Test", "test_metrics")
        ]:

            metrics_df = result[key].copy()

            # Convert the dataframe into one dictionary
            row = {
                "Fold": fold_idx,
                "Dataset": dataset_name
            }

            for col in metrics_df.columns:

                if col != "Dataset":

                    value = metrics_df[col].iloc[0]

                    try:
                        row[col] = float(value)
                    except:
                        pass

            rows.append(row)

    fold_metric_df = pd.DataFrame(rows)

    # =====================================================
    # MEAN + STD
    # =====================================================

    metric_columns = [
        col
        for col in fold_metric_df.columns
        if col not in ["Fold", "Dataset"]
    ]

    mean_df = (
        fold_metric_df
        .groupby("Dataset")[metric_columns]
        .mean()
    )

    std_df = (
        fold_metric_df
        .groupby("Dataset")[metric_columns]
        .std()
    )

    # Create mean ± std table
    summary_df = mean_df.copy()

    for col in metric_columns:

        summary_df[col] = (
            mean_df[col].round(4).astype(str)
            + " ± "
            + std_df[col].round(4).astype(str)
        )

    return fold_metric_df, mean_df, std_df, summary_df
def aggregate_time(fold_results):

    rows = []

    for fold_idx, result in enumerate(fold_results, start=1):

        rows.append({
            "Fold": fold_idx,

            "Train Time": result["train_time"],
            "Validation Evaluation Time": result["val_time"],
            "Test Evaluation Time": result["test_time"],

            "Train Prediction Time": result["train_pred_time"],
            "Validation Prediction Time": result["val_pred_time"],
            "Test Prediction Time": result["test_pred_time"],

            "Total Time": result["total_time"]
        })

    df = pd.DataFrame(rows)

    mean = df.drop(columns="Fold").mean()
    std = df.drop(columns="Fold").std()

    summary = pd.DataFrame({
        "Mean": mean,
        "Std": std
    })

    return df, summary

def aggregate_memory(fold_results):

    rows = []

    for fold_idx, result in enumerate(fold_results, start=1):

        rows.append({
            "Fold": fold_idx,

            "After Model": result["mem_after_model"],
            "After Train": result["mem_after_train"],
            "After Test": result["mem_after_test"],
            "After Validation": result["mem_after_val"],
            "After Prediction": result["mem_end"]
        })

    df = pd.DataFrame(rows)

    mean = df.drop(columns="Fold").mean()
    std = df.drop(columns="Fold").std()

    summary = pd.DataFrame({
        "Mean_MB": mean,
        "Std_MB": std
    })

    return df, summary
def aggregate_confusion_matrices(fold_results):

    train_cm = np.mean(
        [r["train_cm"] for r in fold_results],
        axis=0
    )

    val_cm = np.mean(
        [r["val_cm"] for r in fold_results],
        axis=0
    )

    test_cm = np.mean(
        [r["test_cm"] for r in fold_results],
        axis=0
    )

    return train_cm, val_cm, test_cm
def plot_cm(cm, title):

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    plt.tight_layout()
    plt.show()
from sklearn.metrics import roc_curve, auc

def plot_aggregated_roc(
    fold_results,
    dataset="test"
):

    y_true_all = []
    y_prob_all = []

    for result in fold_results:

        y_true_all.append(
            result[f"{dataset}_y_true"]
        )

        probabilities = result[
            f"{dataset}_probabilities"
        ]

        # Probability of positive class
        y_prob_all.append(
            probabilities[:, 1]
        )

    y_true_all = np.concatenate(y_true_all)
    y_prob_all = np.concatenate(y_prob_all)

    fpr, tpr, thresholds = roc_curve(
        y_true_all,
        y_prob_all
    )

    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))

    plt.plot(
        fpr,
        tpr,
        label=f"5-Fold ROC (AUC = {roc_auc:.4f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    plt.title(
        f"Aggregated ROC Curve - {dataset.capitalize()}"
    )

    plt.legend()
    plt.grid()
    plt.show()

    return roc_auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def plot_multiclass_agg_roc(
    fold_results,
    dataset,
    model_name,
    class_names=None
):

    y_true_all = []
    probabilities_all = []

    for result in fold_results:

        y_true_all.append(
            result[f"{dataset}_y_true"]
        )

        probabilities_all.append(
            result[f"{dataset}_probabilities"]
        )

    y_true_all = np.concatenate(y_true_all)

    probabilities_all = np.concatenate(
        probabilities_all,
        axis=0
    )

    n_classes = probabilities_all.shape[1]

    y_true_binary = label_binarize(
        y_true_all,
        classes=np.arange(n_classes)
    )

    plt.figure(figsize=(8, 6))

    for i in range(n_classes):

        fpr, tpr, _ = roc_curve(
            y_true_binary[:, i],
            probabilities_all[:, i]
        )

        roc_auc = auc(fpr, tpr)

        label = (
            class_names[i]
            if class_names is not None
            else f"Class {i}"
        )

        plt.plot(
            fpr,
            tpr,
            label=f"{label} (AUC={roc_auc:.4f})"
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    plt.title(
        f"5-Fold ROC - {model_name} - {dataset.capitalize()}"
    )

    plt.legend()
    plt.grid()
    plt.tight_layout()

    plt.show()

def aggregate_gpu(fold_results):

    rows = []

    for fold_idx, result in enumerate(fold_results, start=1):

        rows.append({
            "Fold": fold_idx,

            "Before Model": result["gpu_before_model"],
            "After Model": result["gpu_after_model"],
            "After Train": result["gpu_after_train"],
            "After Test": result["gpu_after_test"],
            "After Validation": result["gpu_after_val"],
            "After Prediction": result["gpu_end"]
        })

    df = pd.DataFrame(rows)

    mean = df.drop(columns="Fold").mean()
    std = df.drop(columns="Fold").std()

    summary = pd.DataFrame({
        "Mean_GPU": mean,
        "Std_GPU": std
    })

    return df, summary

In [ ]:
from keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:

# Separate features and target
X = df_combined.drop(columns=['label'])
y = df_combined['label']

# Define number of classes
n_classes = len(np.unique(y))


In [ ]:
def plot_class_distribution(y_train,label="Class Distribution"):
    # Get class distribution from df_combined
    after_df = (
        y_train
        .value_counts()
        .sort_index()               # Keeps labels in numerical order (0,1,2,...)
        .reset_index()
    )

    after_df.columns = ['Attack_type', 'Count']

    # Plot
    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=after_df,
        x='Attack_type',
        y='Count',
        palette='tab10'   # Automatically assigns different colors
    )

    plt.title('Class Distribution before SMOTE')
    plt.xlabel('Attack Type')
    plt.ylabel('Count')

    # Add value labels
    for i, value in enumerate(after_df['Count']):
        plt.text(
            i,
            value + after_df['Count'].max() * 0.01,
            f'{value:,}',
            ha='center',
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

In [ ]:
def train_model(
    model,
    X_train_reshaped, y_train,
    X_val_reshaped, y_val,
    X_test_reshaped, y_test,
    gpu_before_model,
    model_name
):
    # =====================================================
    # MODEL COMPLEXITY
    # =====================================================
    print(f"{model_name} Model Complexity Analysis")


    metrics = model_complexity(
        model,
        input_shape=input_shape
    )

    metrics_df = pd.DataFrame(
        list(metrics.items()),
        columns=["Metric", "Value"]
    )

    metrics_df["Value"] = pd.to_numeric(
        metrics_df["Value"], errors="coerce"
    )

    display(metrics_df.round(4))

    # =====================================================
    # COMPILE
    # =====================================================
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    gpu_after_model = get_gpu_usage()
    mem_after_model = memory_usage_mb()

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    start_train = time.time()

    print(f"Training {model_name}...")

    history = model.fit(
        X_train_reshaped,
        y_train,
        validation_data=(X_val_reshaped, y_val),
        epochs=100,
        batch_size=256,
        callbacks=[
            early_stopping,
            reduce_lr,
            checkpoint
        ],
        verbose=1
    )

    end_train = time.time()

    plot_accuracy(history, model_name)
    plot_loss(history, model_name)

    

    gpu_after_train = get_gpu_usage()
    mem_after_train = memory_usage_mb()

    train_time = end_train - start_train

    print(f"Memory after training: {mem_after_train:.2f} MB")
    print(f"Training time: {train_time:.2f}s")

    # =====================================================
    # TEST EVALUATION
    # =====================================================
    start_test = time.time()

    test_loss, test_accuracy = model.evaluate(
        X_test_reshaped,
        y_test,
        verbose=0
    )

    end_test = time.time()

    gpu_after_test = get_gpu_usage()
    mem_after_test = memory_usage_mb()

    test_time = end_test - start_test

    # =====================================================
    # VALIDATION EVALUATION
    # =====================================================
    start_val = time.time()

    val_loss, val_accuracy = model.evaluate(
        X_val_reshaped,
        y_val,
        verbose=0
    )

    end_val = time.time()

    gpu_after_val = get_gpu_usage()
    mem_after_val = memory_usage_mb()

    val_time = end_val - start_val

    # =====================================================
    # PREDICTIONS
    # =====================================================

    # ---------- TRAIN ----------
    start_pred_train = time.time()

    train_probabilities = model.predict(
        X_train_reshaped,
        verbose=0
    )

    train_pred_classes = np.argmax(
        train_probabilities,
        axis=1
    )

    end_pred_train = time.time()

    train_pred_time = end_pred_train - start_pred_train

    # ---------- VALIDATION ----------
    start_pred_val = time.time()

    val_probabilities = model.predict(
        X_val_reshaped,
        verbose=0
    )

    val_pred_classes = np.argmax(
        val_probabilities,
        axis=1
    )

    end_pred_val = time.time()

    val_pred_time = end_pred_val - start_pred_val

    # ---------- TEST ----------
    start_pred_test = time.time()

    test_probabilities = model.predict(
        X_test_reshaped,
        verbose=0
    )

    test_pred_classes = np.argmax(
        test_probabilities,
        axis=1
    )

    end_pred_test = time.time()

    test_pred_time = end_pred_test - start_pred_test

    mem_end = memory_usage_mb()
    gpu_end = get_gpu_usage()

    # =====================================================
    # METRICS
    # =====================================================

    train_metrics, train_class_metrics, _ = evaluate_dataset(
        model,
        X_train_reshaped,
        y_train,
        f"Training ({model_name})",
        CLASS_NAMES
    )

    val_metrics, val_class_metrics, _ = evaluate_dataset(
        model,
        X_val_reshaped,
        y_val,
        f"Validation ({model_name})",
        CLASS_NAMES
    )

    test_metrics, test_class_metrics, _ = evaluate_dataset(
        model,
        X_test_reshaped,
        y_test,
        f"Test ({model_name})",
        CLASS_NAMES
    )

    # =====================================================
    # CONFUSION MATRICES
    # =====================================================

    train_cm = confusion_matrix(
        y_train,
        train_pred_classes
    )

    val_cm = confusion_matrix(
        y_val,
        val_pred_classes
    )

    test_cm = confusion_matrix(
        y_test,
        test_pred_classes
    )

    # =====================================================
    # MEMORY
    # =====================================================

    memory_summary = pd.DataFrame({
        "Stage": [
            "Start",
            "After model",
            "After train",
            "After test",
            "After validation",
            "After prediction"
        ],

        "Memory_MB": [
            mem_start,
            mem_after_model,
            mem_after_train,
            mem_after_test,
            mem_after_val,
            mem_end
        ]
    })

    # =====================================================
    # GPU
    # =====================================================

    gpu_summary = pd.DataFrame({
        "Stage": [
            "Before model",
            "After model",
            "After train",
            "After test",
            "After validation",
            "After prediction"
        ],

        "GPU_Usage": [
            gpu_before_model,
            gpu_after_model,
            gpu_after_train,
            gpu_after_test,
            gpu_after_val,
            gpu_end
        ]
    })

    # =====================================================
    # TIME
    # =====================================================

    time_summary = pd.DataFrame({
        "Stage": [
            "Train",
            "Test evaluation",
            "Validation evaluation",
            "Train prediction",
            "Validation prediction",
            "Test prediction",
            "Total"
        ],

        "Time_s": [
            train_time,
            test_time,
            val_time,
            train_pred_time,
            val_pred_time,
            test_pred_time,

            train_time
            + test_time
            + val_time
            + train_pred_time
            + val_pred_time
            + test_pred_time
        ]
    })


    print("\nMEMORY: MEAN ± STD "+model_name)
    display(memory_summary)
    print("\nGPU: MEAN ± STD "+model_name)
    display(gpu_summary)
    print("\nTIME: MEAN ± STD "+model_name)
    display(time_summary)

    # =====================================================
    # RETURN EVERYTHING
    # =====================================================

    return {

        # ---------------- METRICS ----------------
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,

        "train_class_metrics": train_class_metrics,
        "val_class_metrics": val_class_metrics,
        "test_class_metrics": test_class_metrics,

        # ---------------- PREDICTIONS ----------------
        "train_y_true": np.asarray(y_train),
        "train_y_pred": train_pred_classes,
        "train_probabilities": train_probabilities,

        "val_y_true": np.asarray(y_val),
        "val_y_pred": val_pred_classes,
        "val_probabilities": val_probabilities,

        "test_y_true": np.asarray(y_test),
        "test_y_pred": test_pred_classes,
        "test_probabilities": test_probabilities,

        # ---------------- CONFUSION MATRICES ----------------
        "train_cm": train_cm,
        "val_cm": val_cm,
        "test_cm": test_cm,

        # ---------------- TIME ----------------
        "train_time": train_time,
        "test_time": test_time,
        "val_time": val_time,

        "train_pred_time": train_pred_time,
        "val_pred_time": val_pred_time,
        "test_pred_time": test_pred_time,

        "total_time": (
            train_time
            + test_time
            + val_time
            + train_pred_time
            + val_pred_time
            + test_pred_time
        ),

        # ---------------- MEMORY ----------------
        "memory_summary": memory_summary,
        "mem_after_model": mem_after_model,
        "mem_after_train": mem_after_train,
        "mem_after_test": mem_after_test,
        "mem_after_val": mem_after_val,
        "mem_end": mem_end,

        # ---------------- GPU ----------------
        # ---------------- GPU ----------------
        "gpu_summary": gpu_summary,

        "gpu_before_model": gpu_before_model,
        "gpu_after_model": gpu_after_model,
        "gpu_after_train": gpu_after_train,
        "gpu_after_test": gpu_after_test,
        "gpu_after_val": gpu_after_val,
        "gpu_end": gpu_end,
         # ---------------- TIME ----------------

        "time_summary": time_summary,

        # ---------------- MODEL ----------------
        "complexity": metrics,

        # ---------------- HISTORY ----------------
        "history": history
    }

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
fold_results = []
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\nFold {fold}/5")

    X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
    y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]

    # ------------------ SPLIT TEMP 50/50 ------------------

    X_temp = X.iloc[temp_idx] if hasattr(X, "iloc") else X[temp_idx]
    y_temp = y.iloc[temp_idx] if hasattr(y, "iloc") else y[temp_idx]

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=42,
        stratify=y_temp
    )

    print(f"Training samples:   {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Test samples:       {len(X_test)}")

    ST=StandardScaler()

    X_train_scaled=ST.fit_transform(X_train)
    X_test_scaled=ST.transform(X_test)
    X_val_scaled=ST.transform(X_val)

    split_labels = (
        ['Train'] * len(y_train) +
        ['Validation'] * len(y_val) +
        ['Test'] * len(y_test)
    )
    y_combined = pd.Series(list(y_train) + list(y_val) + list(y_test), name='Class')
    split_df = pd.DataFrame({'Split': split_labels, 'Class': y_combined})

    # Final shapes
    print(f"Training set shape: X_train: {X_train_scaled.shape}, y_train: {y_train.shape}")
    print(f"Validation set shape: X_val: {X_val_scaled.shape}, y_val: {y_val.shape}")
    print(f"Test set shape: X_test: {X_test_scaled.shape}, y_test: {y_test.shape}")
    plot_class_distribution(y_train,label="Class Distribution before SMOTE")
    # Get class distribution from df_combined
    X_train_scaled,X_val_scaled,X_test_scaled,y_train = scale_dataset(X_train,X_val,X_test,y_train,y_val,y_test)
    # Reshape input data for LSTM
    plot_class_distribution(y_train,label="Class Distribution after SMOTE")
    print("Reshaping input data for LSTM...")
    X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_val_reshaped = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
    X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))
    # One-hot encode the labelstrai
    print("One-hot encoding labels...")
    y_train_multi_encoded = to_categorical(y_train, num_classes=n_classes)
    y_val_multi_encoded = to_categorical(y_val, num_classes=n_classes)
    y_test_multi_encoded = to_categorical(y_test, num_classes=n_classes)
    input_shape = X_train_reshaped.shape[1:]
    print("Data preprocessing complete.")

    CLASS_NAMES =list(set(y_train))
    NUM_CLASSES =len(CLASS_NAMES)
    print('*'*80)
    print("*******************  MODEL********************")
    print('*'*80)
    gpu_before_model = get_gpu_usage()
    model = create_model(
        input_shape=input_shape,
        lstm_units=16,
        l2_reg=1e-4,
        dense1=32,
        dense2=16,
        n_classes=n_classes
    )
    result = train_model(model,X_train_reshaped,y_train,X_val_reshaped, y_val,X_test_reshaped, y_test,gpu_before_model,"BSLA-ANN")
    fold_results.append(result)
    

In [ ]:
# =====================================================
# 5-FOLD FINAL REPORT
# =====================================================

print("=" * 80)
print("5-FOLD CROSS-VALIDATION FINAL RESULTS")
print("=" * 80)


# -----------------------------------------------------
# METRICS
# -----------------------------------------------------

fold_metric_df, mean_metrics, std_metrics, metrics_summary = (
    aggregate_fold_metrics(fold_results)
)

print("\nMETRICS: MEAN ± STD")
display(metrics_summary)


# -----------------------------------------------------
# TIME
# -----------------------------------------------------

time_folds, time_summary = aggregate_time(
    fold_results
)

print("\nTIME: MEAN ± STD" )
display(time_summary.round(4))


# -----------------------------------------------------
# MEMORY
# -----------------------------------------------------

memory_folds, memory_summary = aggregate_memory(
    fold_results
)

print("\nMEMORY: MEAN ± STD")
display(memory_summary.round(2))


# -----------------------------------------------------
# GPU
# -----------------------------------------------------

gpu_folds, gpu_summary = aggregate_gpu(
    fold_results
)

print("\nGPU USAGE: MEAN ± STD")
display(gpu_summary.round(2))


# -----------------------------------------------------
# CONFUSION MATRICES
# -----------------------------------------------------

train_cm, val_cm, test_cm = (
    aggregate_confusion_matrices(
        fold_results
    )
)

print("\nCONFUSION MATRICES")

plot_cm(
    train_cm,
    "5-Fold Aggregated Confusion Matrix   - Train"
)

plot_cm(
    val_cm,
    "5-Fold Aggregated Confusion Matrix   - Validation"
)

plot_cm(
    test_cm,
    "5-Fold Aggregated Confusion Matrix   - Test"
)


# -----------------------------------------------------
# ROC
# -----------------------------------------------------

plot_multiclass_agg_roc(
    fold_results,
    dataset="train",
    model_name="BSLA-ANN",
    class_names=CLASS_NAMES
)

plot_multiclass_agg_roc(
    fold_results,
    dataset="val",
    model_name="BSLA-ANN",
    class_names=CLASS_NAMES
)

plot_multiclass_agg_roc(
    fold_results,
    dataset="test",
    model_name="BSLA-ANN",
    class_names=CLASS_NAMES
)

In [ ]:
"done"